# Micro-Hay Orthogonal Factorial 11

Training da zero preregistrato `2 x 2`: **GRU vs CausalConv1d+GRU** e **MSE vs MSE+MR-STFT**. Le quattro condizioni condividono dataset, seed, batch, finestre, budget massimo, stopping policy e criterio di selezione.

La metrica primaria e il soma RMSE sugli eventi di validation. Global NRMSE, stati lenti e subthreshold sono guardrail. Il test non viene mai aperto. Sono inclusi progress/ETA, resume, checkpoint best/last e Failure Atlas completo per ogni run.

In [ ]:
from pathlib import Path
import subprocess, sys

REPOSITORY_URL = 'https://github.com/Zagred47/LearningSingleCompartiment.git'

def valid_repository(path):
    path = Path(path)
    return ((path / 'pyproject.toml').is_file()
            and (path / 'src/hay_single_compartment').is_dir()
            and (path / 'notebooks/micro_orthogonal_factorial_11.py').is_file())

candidates = [Path('/kaggle/working/LearningSingleCompartiment')]
candidates += [p.parent for p in Path('/kaggle/working').glob('**/pyproject.toml')]
candidates += [p.parent for p in Path('/kaggle/input').glob('**/pyproject.toml')]
REPO_ROOT = next((p.resolve() for p in candidates if valid_repository(p)), None)
if REPO_ROOT is None:
    base = Path('/kaggle/working/LearningSingleCompartiment_factorial')
    target, suffix = base, 1
    while target.exists():
        target = Path(f'{base}_{suffix}'); suffix += 1
    subprocess.check_call(['git', 'clone', '--depth', '1', REPOSITORY_URL, str(target)])
    REPO_ROOT = target.resolve()
src = str(REPO_ROOT / 'src')
if src not in sys.path: sys.path.insert(0, src)
print('Repository:', REPO_ROOT)

## Dataset e configurazione

Monta `hay_micro_4c_event_enriched_v2.h5` come Kaggle Input. Il dataset viene soltanto letto: non viene rigenerato. I default eseguono quattro run da massimo 30 epoche.

In [ ]:
import os
from pathlib import Path

datasets = sorted(Path('/kaggle/input').glob('**/*.h5'))
print('HDF5 disponibili:')
for path in datasets: print(' ', path)

# Decommenta soltanto se la discovery automatica sceglie il file sbagliato:
# os.environ['HAY_FACTORIAL_DATASET'] = '/kaggle/input/.../hay_micro_4c_event_enriched_v2.h5'
os.environ['HAY_FACTORIAL_OUTPUT'] = '/kaggle/working/hay_micro_orthogonal_factorial_11'

# Preflight tecnico opzionale (NON usare per il risultato scientifico):
# os.environ['HAY_FACTORIAL_EPOCHS'] = '1'
# os.environ['HAY_FACTORIAL_MINIMUM_EPOCHS'] = '1'
# os.environ['HAY_FACTORIAL_RUNS'] = 'gru_mse'
# os.environ['HAY_FACTORIAL_MAX_TRAIN_TRAJECTORIES'] = '2'
# os.environ['HAY_FACTORIAL_MAX_VALIDATION_TRAJECTORIES'] = '1'
# os.environ['HAY_FACTORIAL_WINDOWS_PER_EPOCH'] = '6'
# os.environ['HAY_FACTORIAL_CONTEXT_STEPS'] = '256'
# Per esercitare MR-STFT già nell'epoca 1 del solo preflight tecnico:
# os.environ['HAY_FACTORIAL_SPECTRAL_WARMUP_EPOCHS'] = '0'
# os.environ['HAY_FACTORIAL_SPECTRAL_CURRICULUM_EPOCHS'] = '1'

# Default: lo ZIP include i best checkpoint ma esclude i last checkpoint di resume.
# os.environ['HAY_FACTORIAL_DOWNLOAD_LAST_CHECKPOINTS'] = '1'

In [ ]:
import runpy
result = runpy.run_path(str(REPO_ROOT / 'notebooks/micro_orthogonal_factorial_11.py'))
OUTPUT_DIR = Path(result['OUTPUT'])
ZIP_PATH = Path(result['ZIP_PATH'])
print('Output:', OUTPUT_DIR)
print('ZIP:', ZIP_PATH)

In [ ]:
import pandas as pd
from IPython.display import display
comparison = pd.read_csv(OUTPUT_DIR / 'validation_comparison.csv')
display(comparison.sort_values('event_soma_rmse_mV'))
effects_path = OUTPUT_DIR / 'factorial_effects.csv'
if effects_path.exists(): display(pd.read_csv(effects_path))
display(pd.read_json(OUTPUT_DIR / 'preflight_decision.json', typ='series'))

In [ ]:
import matplotlib.pyplot as plt
from IPython.display import Image, display

fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))
comparison.plot.bar(x='run', y='event_soma_rmse_mV', ax=axes[0], legend=False, title='Event soma RMSE')
comparison.plot.bar(x='run', y='mean_state_nrmse', ax=axes[1], legend=False, title='Global state NRMSE')
comparison.plot.bar(x='run', y='spike_recall', ax=axes[2], legend=False, title='Spike recall')
for axis in axes: axis.grid(axis='y', alpha=.25)
plt.tight_layout(); plt.show()

for run in comparison['run']:
    path = OUTPUT_DIR / 'failure_atlases' / run / 'teacher_centered_spike_waveform.png'
    if path.exists():
        print(run); display(Image(filename=str(path)))

In [ ]:
# Download: FileLink sempre disponibile; Blob automatico solo se lo ZIP non e enorme.
from IPython.display import FileLink, Javascript, display
import base64, os
display(FileLink(str(ZIP_PATH)))
size_mib = ZIP_PATH.stat().st_size / 2**20
if size_mib <= 80 or os.environ.get('HAY_FACTORIAL_FORCE_BLOB_DOWNLOAD') == '1':
    encoded = base64.b64encode(ZIP_PATH.read_bytes()).decode('ascii')
    filename = ZIP_PATH.name
    display(Javascript(f'''
    const binary = atob('{encoded}');
    const bytes = new Uint8Array(binary.length);
    for (let i = 0; i < binary.length; i++) bytes[i] = binary.charCodeAt(i);
    const blob = new Blob([bytes], {{type: 'application/zip'}});
    const url = URL.createObjectURL(blob);
    const anchor = document.createElement('a');
    anchor.href = url; anchor.download = '{filename}';
    document.body.appendChild(anchor); anchor.click(); anchor.remove();
    setTimeout(() => URL.revokeObjectURL(url), 60000);
    '''))
    print('Download Blob avviato:', ZIP_PATH, f'({size_mib:.1f} MiB)')
else:
    print(f'ZIP da {size_mib:.1f} MiB: usa il FileLink sopra o il pannello Files di Kaggle.')